<a href="https://colab.research.google.com/github/Adepuharshavardhan2001/AI-Chatbot-Mentor/blob/main/vpr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ortools folium

In [ ]:
import pandas as pd

locations = [
    ("Depot", 17.3850, 78.4867),
    ("Hitech City", 17.4435, 78.3772),
    ("Ameerpet", 17.4375, 78.4483),
    ("Kukatpally", 17.4948, 78.3996),
    ("Banjara Hills", 17.4126, 78.4482),
    ("Gachibowli", 17.4401, 78.3489),
    ("Secunderabad", 17.4399, 78.4983),
    ("Madhapur", 17.4486, 78.3915),
    ("Begumpet", 17.4448, 78.4625)
]

df = pd.DataFrame(locations, columns=["location", "lat", "lon"])

# Increase demand slightly to force multiple vehicles
df["demand"] = [0, 3, 2, 4, 3, 2, 3, 2, 2]

df

,location,lat,lon,demand
0,Depot,17.3850,78.4867,0
1,Hitech City,17.4435,78.3772,3
2,Ameerpet,17.4375,78.4483,2
3,Kukatpally,17.4948,78.3996,4
4,Banjara Hills,17.4126,78.4482,3
5,Gachibowli,17.4401,78.3489,2
6,Secunderabad,17.4399,78.4983,3
7,Madhapur,17.4486,78.3915,2
8,Begumpet,17.4448,78.4625,2


In [ ]:
import folium
from IPython.display import display

m = folium.Map(location=[df['lat'].mean(), df['lon'].mean()], zoom_start=12)

for _, row in df.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=f"{row['location']} (Demand: {row['demand']})"
    ).add_to(m)

display(m)

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist

coords = df[['lat', 'lon']].values
distance_matrix = cdist(coords, coords, metric='euclidean')

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

num_vehicles = 3
vehicle_capacities = [8, 8, 8]
depot = 0

manager = pywrapcp.RoutingIndexManager(len(distance_matrix), num_vehicles, depot)
routing = pywrapcp.RoutingModel(manager)

In [ ]:
def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return int(distance_matrix[from_node][to_node] * 1000)

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

In [ ]:
routing.AddDimension(
    transit_callback_index,
    0,
    500,   # max distance
    True,
    "Distance"
)

True

In [ ]:
demands = df["demand"].tolist()

def demand_callback(from_index):
    node = manager.IndexToNode(from_index)
    return demands[node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)

In [ ]:
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    vehicle_capacities,
    True,
    "Capacity"
)

True

In [ ]:
search_parameters = pywrapcp.DefaultRoutingSearchParameters()

search_parameters.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

solution = routing.SolveWithParameters(search_parameters)

if solution:
    print("Solution found ✅")
else:
    print("No solution ❌")

Solution found ✅


In [ ]:
def print_routes():
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route = []

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            route.append(df.loc[node, "location"])
            index = solution.Value(routing.NextVar(index))

        route.append("Depot")
        print(f"\n🚚 Vehicle {vehicle_id}")
        print("Route:", " → ".join(route))

print_routes()


🚚 Vehicle 0
Route: Depot → Ameerpet → Kukatpally → Madhapur → Depot

🚚 Vehicle 1
Route: Depot → Banjara Hills → Hitech City → Gachibowli → Depot

🚚 Vehicle 2
Route: Depot → Begumpet → Secunderabad → Depot


In [ ]:
colors = ['red', 'blue', 'green']

import folium
from IPython.display import display

# Create fresh map (important)
m = folium.Map(location=[df['lat'].mean(), df['lon'].mean()], zoom_start=12)

# Add markers again
for _, row in df.iterrows():
    folium.Marker(
        [row['lat'], row['lon']],
        popup=row['location']
    ).add_to(m)

# Draw routes
for vehicle_id in range(num_vehicles):
    index = routing.Start(vehicle_id)
    route_coords = []

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)
        route_coords.append([df.loc[node, 'lat'], df.loc[node, 'lon']])
        index = solution.Value(routing.NextVar(index))

    # return to depot
    route_coords.append([df.loc[0, 'lat'], df.loc[0, 'lon']])

    # draw line
    folium.PolyLine(
        route_coords,
        color=colors[vehicle_id],
        weight=5,
        opacity=0.8
    ).add_to(m)

display(m)

In [ ]:
for vehicle_id in range(num_vehicles):
    index = routing.Start(vehicle_id)
    step = 0

    while not routing.IsEnd(index):
        node = manager.IndexToNode(index)

        folium.Marker(
            [df.loc[node, 'lat'], df.loc[node, 'lon']],
            popup=f"Vehicle {vehicle_id} | Step {step} | {df.loc[node, 'location']}",
            icon=folium.DivIcon(html=f"""<div style="font-size: 12pt; color : black">{step}</div>""")
        ).add_to(m)

        index = solution.Value(routing.NextVar(index))
        step += 1

In [ ]:
def print_routes_with_distance():
    total_distance = 0

    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        route_distance = 0

        while not routing.IsEnd(index):
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        print(f"Vehicle {vehicle_id} Distance: {route_distance}")
        total_distance += route_distance

    print(f"\nTotal Distance: {total_distance}")

print_routes_with_distance()

Vehicle 0 Distance: 300
Vehicle 1 Distance: 300
Vehicle 2 Distance: 156

Total Distance: 756


In [ ]:
def print_load():
    for vehicle_id in range(num_vehicles):
        index = routing.Start(vehicle_id)
        load = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            load += demands[node]
            index = solution.Value(routing.NextVar(index))

        print(f"Vehicle {vehicle_id} Load: {load}")

print_load()

Vehicle 0 Load: 8
Vehicle 1 Load: 8
Vehicle 2 Load: 5


In [ ]:
for vehicle_id in range(num_vehicles):
    folium.Marker(
        [df.loc[0, 'lat'], df.loc[0, 'lon']],
        popup=f"Vehicle {vehicle_id} Start",
        icon=folium.Icon(color='black')
    ).add_to(m)

In [ ]:
m.save("vrp_routes.html")

In [ ]:
folium.PolyLine(
    route_coords,
    color=colors[vehicle_id],
    weight=6,
    opacity=0.9,
    tooltip=f"Vehicle {vehicle_id}"
).add_to(m)

In [ ]:
m